# Feedforward ensemble of source rules for bed prediction

This notebook learns nonnegative weights that sum to one for predictions from XGBoost models trained on four deterministic count-supplementation rules. A small feedforward network can vary the weights by facility. It learns from errors on out-of-fold reported bed counts. Compare it with the best individual rule, equal weights, inverse-error weights, and a learned fixed blend before choosing a method.

Use one feature tier for every rule. The default matches the facility-plus-population tier in the [rule comparison](../../max-supplemented-data/experiments/capacity/RULE_CLEANING_COMPARISON.md). The notebook consumes the private `rule-cleaning-v2` checkpoint. It has no saved execution results. Running it requires Python 3.11 or later, NumPy, pandas, SciPy, scikit-learn, XGBoost and CPU PyTorch. Set the optional paths below if the checkout or authorized local data are elsewhere.

In [ ]:
from copy import deepcopy
from collections import defaultdict
from pathlib import Path
import os
import sys
import json

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.optimize import linprog
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
import torch
from torch import nn
import xgboost as xgb

candidate_roots = [Path.cwd(), *Path.cwd().parents]
configured_root = os.getenv("DATATHON_RULE_ENSEMBLE_REPO")
if configured_root:
    candidate_roots.insert(0, Path(configured_root))
repo = next((root for root in candidate_roots
             if (root / "max-supplemented-data/experiments/capacity/train_evaluate.py").is_file()), None)
if repo is None:
    raise FileNotFoundError("Set DATATHON_RULE_ENSEMBLE_REPO to the checkout.")
sys.path.insert(0, str(repo / "max-supplemented-data/experiments/capacity"))
from train_evaluate import (
    PUBLISHED_METHODS, calibrated_radius, convert_beds, district_clusters, encoder_matrix,
    fingerprint, fit_encoding, interval_metrics, load_inputs, paired_interval, receipt, score, split_masks,
)

run = Path(os.getenv("DATATHON_RULE_ENSEMBLE_RUN",
                     str(repo / "max-supplemented-data/local/rule-cleaning-v2")))
FEATURE_TIER = "population"  # facility_only, socioeconomic, population, socioeconomic_population
SEED = 217
FOLDS = 5
EXPORT = False  # Writes a new local experiment directory only when enabled.
torch.manual_seed(SEED)
torch.set_num_threads(2)

## 1. Check the input and identify duplicate rule predictions

A rule's bed target and doctor feature come from its own version of the cleaned data. Where two rules produce identical model inputs and use the same XGBoost configuration, they share one ensemble input. The current newest, same-year and two-year rules are expected to share a column. Their combined weight is identifiable. The weight split among identical rule names is reported evenly.

In [ ]:
metrics_path = run / "training/metrics.json"
metrics = json.loads(metrics_path.read_text())
frame, groups = load_inputs(run / "inputs")
for name, digest in metrics["input_sha256"].items():
    if receipt(run / "inputs" / name) != digest:
        raise ValueError(f"Saved models and assembled input disagree for {name}.")

policies = {key: value for key, value in metrics["source_policies"].items() if key != "raw_fdr"}
if set(policies.values()) != {"ranked", "newest", "latest_year_mean", "recent_two_year_mean"}:
    raise ValueError("Use the completed four-policy comparison.")
arm_by_policy = {
    entry["iteration"]: entry for entry in metrics["runs"]
    if entry["feature_set"] == FEATURE_TIER and entry["iteration"] in policies
}
if arm_by_policy.keys() != policies.keys():
    raise ValueError("One archived model and feature tier is required for each rule.")
tune, interval, test = split_masks(frame)
for name, mask in {"validation": tune, "interval_calibration": interval, "test": test}.items():
    expected = metrics["holdout"][name]["sha256_facility_and_observed_count"]
    if fingerprint(frame, mask) != expected:
        raise ValueError(f"The saved {name} facility and label fingerprint changed.")

def model_input_columns(iteration):
    return [f"capacity_{iteration}", f"numberDoctors_{iteration}"]

def same_inputs(a, b):
    a_cols, b_cols = model_input_columns(a), model_input_columns(b)
    av = frame[a_cols].to_numpy(dtype=float)
    bv = frame[b_cols].to_numpy(dtype=float)
    ac, bc = arm_by_policy[a], arm_by_policy[b]
    return (np.array_equal(av, bv, equal_nan=True)
            and ac["chosen_config"] == bc["chosen_config"]
            and ac["features"] == bc["features"])

equivalents = defaultdict(list)
for policy in policies:
    representative = next((key for key in equivalents if same_inputs(key, policy)), policy)
    equivalents[representative].append(policy)
representatives = list(equivalents)
column_labels = [" / ".join(policies[item] for item in equivalents[key]) for key in representatives]
display(pd.DataFrame({"prediction_column": column_labels,
                      "rule_count": [len(equivalents[key]) for key in representatives]}))

## 2. Generate out-of-fold predictions for the network

Use only the original training partition. Score each out-of-fold branch against its original reported bed count. Remove its entire district from each fold's base-model fit, including supplementary training rows in that district. Each fold fits its input encoding from scratch. The source rule stays fixed. Reuse the archived configuration and stopping round for its chosen feature tier. Facility identity and actual test error never enter the network.

In [ ]:
branch_groups = district_clusters(frame)
base_train = (frame.split.eq("train") & frame.training_eligible).to_numpy()
oof_ids = np.flatnonzero(base_train & frame.evaluation_eligible.to_numpy()
                         & frame.capacity_observed.notna().to_numpy())
y_observed = frame.capacity_observed.to_numpy(dtype=float)

def fitted_parameters(arm):
    active = arm["chosen_config"]
    return next(trial["parameters"] for trial in arm["tuning"] if
                all(trial[field] == active[field] for field in ("scale", "depth", "best_round")))

def predictions(booster, matrix, arm):
    cfg = arm["chosen_config"]
    raw = booster.predict(xgb.DMatrix(matrix, nthread=2), iteration_range=(0, cfg["best_round"]))
    return convert_beds(raw, cfg["scale"])

def fit_fold(iteration, train_mask, held_index):
    arm = arm_by_policy[iteration]
    targets = frame[f"capacity_{iteration}"]
    supported = train_mask & targets.notna().to_numpy()
    methods = frame.loc[supported, f"capacity_method_{iteration}"]
    if not methods.isin(PUBLISHED_METHODS).all():
        raise ValueError("A fold contains a non-published capacity training label.")
    alias = {"numberDoctors": f"numberDoctors_{iteration}"}
    encoding = fit_encoding(frame.loc[supported], arm["features"], groups["feature_types"], alias)
    x_build = encoder_matrix(frame.loc[supported], encoding)
    y_build = targets.loc[supported].to_numpy(dtype=float)
    if arm["chosen_config"]["scale"] == "log1p_squared":
        y_build = np.log1p(y_build)
    model = xgb.train(fitted_parameters(arm), xgb.DMatrix(x_build, label=y_build, nthread=2),
                      num_boost_round=arm["chosen_config"]["best_round"], verbose_eval=False)
    return predictions(model, encoder_matrix(frame.iloc[held_index], encoding), arm)

oof = np.full((len(oof_ids), len(representatives)), np.nan)
for _, held in GroupKFold(n_splits=FOLDS).split(oof_ids, groups=branch_groups[oof_ids]):
    positions = oof_ids[held]
    blocked = np.unique(branch_groups[positions])
    model_train = base_train & ~np.isin(branch_groups, blocked)
    if model_train[positions].any():
        raise ValueError("An out-of-fold branch leaked into its base models.")
    for column, policy in enumerate(representatives):
        oof[held, column] = fit_fold(policy, model_train, positions)
if not np.isfinite(oof).all():
    raise ValueError("The out-of-fold prediction matrix is incomplete.")
y_oof = y_observed[oof_ids]
display(pd.DataFrame({"rule": column_labels,
                      "out_of_fold_mae_beds": np.abs(oof - y_oof[:, None]).mean(axis=0)}))

## 3. Fit a small network and select the weighting method

Use a district-grouped development split of the out-of-fold predictions. The network reads log-transformed predictions and their disagreement, then produces a softmax over the rule columns. Train it to reduce mean absolute error in beds. Use out-of-fold development MAE to stop training and to compare the gated network with simpler weighting methods. A single surviving rule bypasses fitting because its weight is necessarily one. The fixed blend minimizes training MAE under the same nonnegative, sum-to-one constraint and uses identical weights at every facility. It provides a direct comparison with the network’s facility-specific weights.

In [ ]:
meta_train, meta_dev = next(GroupShuffleSplit(n_splits=1, test_size=.20, random_state=SEED)
                            .split(oof, groups=branch_groups[oof_ids]))
if set(branch_groups[oof_ids[meta_train]]) & set(branch_groups[oof_ids[meta_dev]]):
    raise ValueError("Districts must be disjoint in the ensemble development split.")

def context(values):
    logged = np.log1p(values)
    return np.column_stack((logged, logged.max(axis=1) - logged.min(axis=1)))

center = context(oof[meta_train]).mean(axis=0)
spread = context(oof[meta_train]).std(axis=0)
spread[spread < 1e-8] = 1
quality = np.abs(oof[meta_train] - y_oof[meta_train, None]).mean(axis=0)
inverse = 1 / np.maximum(quality, 1e-8)
inverse /= inverse.sum()
one_rule = np.eye(len(representatives))[int(np.argmin(quality))]
equal = np.full(len(representatives), 1 / len(representatives))

# Each training row gets a nonnegative variable for its absolute residual.
blend_inputs, blend_actual = oof[meta_train], y_oof[meta_train]
n_samples, n_rules = blend_inputs.shape
residual_variables = -sparse.eye(n_samples, format="csr")
fixed_fit = linprog(
    c=np.r_[np.zeros(n_rules), np.ones(n_samples) / n_samples],
    A_ub=sparse.vstack([
        sparse.hstack([blend_inputs, residual_variables]),
        sparse.hstack([-blend_inputs, residual_variables]),
    ], format="csr"),
    b_ub=np.r_[blend_actual, -blend_actual],
    A_eq=sparse.csr_matrix(np.r_[np.ones(n_rules), np.zeros(n_samples)].reshape(1, -1)),
    b_eq=[1.0],
    bounds=(0, None),
    method="highs",
)
if not fixed_fit.success:
    raise ValueError(f"Could not fit constant blend weights: {fixed_fit.message}")
fixed = fixed_fit.x[:n_rules]

def tensor(values):
    return torch.as_tensor(values, dtype=torch.float32)

gate = nn.Sequential(nn.Linear(len(representatives) + 1, 8), nn.ReLU(),
                     nn.Linear(8, len(representatives)), nn.Softmax(dim=1))

def gate_weights(values):
    if len(representatives) == 1:
        return np.ones((len(values), 1))
    with torch.no_grad():
        gate.eval()
        return gate(tensor((context(values) - center) / spread)).numpy()

if len(representatives) > 1:
    optimizer = torch.optim.AdamW(gate.parameters(), lr=0.01, weight_decay=0.02)
    tr_x = tensor((context(oof[meta_train]) - center) / spread)
    tr_beds = tensor(oof[meta_train])
    tr_y = tensor(y_oof[meta_train])
    best_loss, best_state, best_epoch = float("inf"), None, 0
    for epoch in range(250):
        gate.train()
        optimizer.zero_grad()
        estimated = (gate(tr_x) * tr_beds).sum(dim=1)
        loss = (estimated - tr_y).abs().mean() / max(float(np.median(y_oof[meta_train])), 1)
        loss.backward()
        optimizer.step()
        estimated_dev = (gate_weights(oof[meta_dev]) * oof[meta_dev]).sum(axis=1)
        dev_loss = np.abs(estimated_dev - y_oof[meta_dev]).mean()
        if dev_loss < best_loss - 1e-6:
            best_loss, best_state, best_epoch = dev_loss, deepcopy(gate.state_dict()), epoch
        if epoch - best_epoch >= 30:
            break
    if best_state is None:
        raise ValueError("Ensemble optimization did not produce a finite result.")
    gate.load_state_dict(best_state)

methods = {"best_single_rule": lambda values: np.broadcast_to(one_rule, values.shape),
           "equal_blend": lambda values: np.broadcast_to(equal, values.shape),
           "inverse_oof_mae": lambda values: np.broadcast_to(inverse, values.shape),
           "learned_fixed_blend": lambda values: np.broadcast_to(fixed, values.shape),
           "feedforward_gate": gate_weights}

def rule_blend(method, values):
    weights = methods[method](values)
    if np.any(weights < 0) or not np.allclose(weights.sum(axis=1), 1):
        raise ValueError("Invalid combination weights.")
    return np.maximum(1, np.rint((weights * values).sum(axis=1))).astype(int)

development_mae = {method: np.abs(rule_blend(method, oof[meta_dev]) - y_oof[meta_dev]).mean()
                   for method in methods}
winner = min(methods, key=development_mae.get)
display(pd.DataFrame({"method": list(methods), "out_of_fold_development_mae_beds": list(development_mae.values())}))
print("Selected before test:", winner)

## 4. Evaluate the selected blend on the shared test

Use the archived full-training base boosters and their encoders to predict the calibration and test branches. Estimate interval widths from calibration residuals. Report the test error for all methods and paired differences from the best single rule. The shared historical test has been used earlier in the project. A later independent validation should use newly adjudicated bed counts.

In [ ]:
def archived_values(indices):
    columns = []
    for policy in representatives:
        arm = arm_by_policy[policy]
        encoding = json.loads((run / "training" / arm["encoding_file"]).read_text())
        booster = xgb.Booster()
        booster.load_model(run / "training" / arm["model_file"])
        columns.append(predictions(booster, encoder_matrix(frame.iloc[indices], encoding), arm))
    return np.column_stack(columns)

test_pos, cal_pos = np.flatnonzero(test), np.flatnonzero(interval)
test_values = archived_values(test_pos)
cal_values = archived_values(cal_pos)
test_actual = y_observed[test_pos]
reference = np.abs(rule_blend("best_single_rule", test_values) - test_actual)
results = []
for method in methods:
    prediction = rule_blend(method, test_values)
    cal_prediction = rule_blend(method, cal_values)
    radius = calibrated_radius(y_observed[cal_pos], cal_prediction)
    results.append({"method": method, "selected": method == winner,
                    **score(test_actual, prediction),
                    **interval_metrics(test_actual, prediction, radius),
                    "calibration_radius_beds": radius,
                    **paired_interval(reference, np.abs(prediction - test_actual), branch_groups[test_pos])})
display(pd.DataFrame(results))

## 5. Return the combination weights and optional prediction table

For equal-input rules, split the shared column's weight evenly when displaying a weight per named rule. The learned allocation is between distinct model outputs. Publish development-average weights and facility-level weights. Any prediction table is restricted to routable branches that every rule allows to be estimated. The optional export writes a new research directory and leaves the existing capacity release intact.

In [ ]:
def weights_by_rule(weights):
    columns = {}
    for position, representative in enumerate(representatives):
        linked = equivalents[representative]
        for alias in linked:
            columns[policies[alias]] = weights[:, position] / len(linked)
    return pd.DataFrame(columns)

for method in dict.fromkeys((winner, "feedforward_gate", "learned_fixed_blend")):
    by_rule = weights_by_rule(methods[method](oof[meta_dev]))
    display(pd.DataFrame({"source_rule": by_rule.columns, "development_mean_weight": by_rule.mean().values})
            .assign(method=method))

missing_mask = frame.capacity_observed.isna().to_numpy(copy=True)
for iteration in policies:
    missing_mask &= frame[f"imputation_eligible_{iteration}"].to_numpy(dtype=bool)
estimate_positions = np.flatnonzero(missing_mask)
if len(estimate_positions):
    base_predictions = archived_values(estimate_positions)
    prediction = rule_blend(winner, base_predictions)
    radius = next(row["calibration_radius_beds"] for row in results if row["method"] == winner)
    output = weights_by_rule(methods[winner](base_predictions))
    output.insert(0, "facility_id", frame.index[estimate_positions])
    output["capacity_estimated_beds"] = prediction
    output["lower_beds"] = np.maximum(0, prediction - radius).astype(int)
    output["upper_beds"] = (prediction + radius).astype(int)
else:
    output = pd.DataFrame(columns=["facility_id", *policies.values(), "capacity_estimated_beds",
                                   "lower_beds", "upper_beds"])

if EXPORT:
    dest = run / f"feedforward-ensemble-{FEATURE_TIER}"
    dest.mkdir(exist_ok=False)
    pd.DataFrame(results).to_csv(dest / "test_metrics.csv", index=False)
    output.to_csv(dest / "candidate_estimates_with_weights.csv", index=False)
    torch.save({"gate_weights": gate.state_dict() if len(representatives) > 1 else {},
                "center": tensor(center), "spread": tensor(spread), "feature_tier": FEATURE_TIER,
                "rules": {key: [policies[item] for item in equivalents[key]] for key in representatives},
                "source_model_checksums": {key: receipt(run / "training" / arm_by_policy[key]["model_file"])
                                           for key in representatives},
                "selected_method": winner, "equal_weights": equal.tolist(),
                "best_single_weights": one_rule.tolist(), "inverse_oof_weights": inverse.tolist(),
                "learned_fixed_weights": fixed.tolist()},
               dest / "gate.pt")
print("Eligible branches for the ensemble estimate:", len(output))